# AgriSmart AI — EfficientNet-B2 Training on Colab GPU

**Self-contained training notebook. Run cells top-to-bottom.**

### Before running
1. `Runtime → Change runtime type → T4 GPU` (free tier)
2. Run all cells in order — do NOT skip cells
3. After training, Cell 19 auto-downloads model weights + results

### Pipeline
1. GPU check
2. Clone repo + install deps
3. Configure paths
4. Define 38→28 class mapping (with filesystem-safe sanitization)
5. Direct PlantVillage dataset download via Hugging Face `data.zip` (RGB color)
6. Inspect downloaded raw dataset
7. Apply 38→28 mapping (sanitized folder names, original filenames preserved)
8. Create 80/10/10 **leaf-group isolated** leakage-safe split
9. **Verify processed dataset** (class counts, file/leaf overlap checks, missing/empty dirs)
10. **Diagnostic paths cell**
11. **Assert TRAIN_DIR exists and is non-empty** before ImageFolder
12. Define transforms
13. Build DataLoaders + verify class mapping
14. Verify DataLoader batch shapes
15. Build model
16. Training loop (2-stage: warmup freeze → full fine-tune)
17. Evaluation (confusion matrix, per-class metrics)
18. Save classes.json
19. Download all results

## Cell 1 — Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU found. Go to Runtime > Change runtime type > GPU before continuing.')

## Cell 2 — Clone Repository and Install Dependencies

In [ ]:
import subprocess, sys

# Clone the AgriSmart-AI repo
REPO_URL  = 'https://github.com/Parrthiv125/AgriSmart-AI.git'
REPO_DIR  = '/content/agrismart'

result = subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    # Repo already exists — just pull latest
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo already present — pulled latest.')
else:
    print(result.stdout or 'Cloned successfully.')

# Install required packages (uses huggingface_hub for data.zip download)
!pip install -q timm huggingface_hub scikit-learn tqdm seaborn matplotlib

print('Dependencies installed.')

## Cell 3 — Configuration

**Note:** Paths are NOT pre-created here. Directories are created only at the step that writes into them, so an empty TRAIN_DIR can never exist before the split runs.

In [ ]:
from pathlib import Path

# ── Root paths ─────────────────────────────────────────────────────────────────
ROOT          = Path(REPO_DIR)                  # /content/agrismart
RAW_DIR       = ROOT / 'data' / 'raw' / 'plantvillage'  # raw PlantVillage images
MAPPED_DIR    = ROOT / 'data' / 'mapped'        # after 38->28 class mapping
PROCESSED_DIR = ROOT / 'data' / 'processed'    # after train/val/test split
TRAIN_DIR     = PROCESSED_DIR / 'train'
VAL_DIR       = PROCESSED_DIR / 'val'
TEST_DIR      = PROCESSED_DIR / 'test'
MODELS_DIR    = ROOT / 'models'
EVAL_DIR      = ROOT / 'evaluation' / 'results'

# These are created as needed
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODELS_DIR / 'agrismart_best.pth'
LAST_MODEL_PATH = MODELS_DIR / 'agrismart_last.pth'
CLASSES_JSON    = MODELS_DIR / 'classes.json'
EXPERIMENT_LOG  = ROOT / 'experiments.csv'

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'efficientnet_b2'
PRETRAINED  = True
IMAGE_SIZE  = 260

# ── Training hyperparameters ────────────────────────────────────────────────────
SEED                    = 42
BATCH_SIZE              = 32
NUM_EPOCHS              = 25
LEARNING_RATE           = 1e-4
WEIGHT_DECAY            = 1e-4
WARMUP_EPOCHS           = 3       # freeze backbone for first N epochs
EARLY_STOPPING_PATIENCE = 7
USE_AMP                 = True
NUM_WORKERS             = 4
TRAIN_RATIO             = 0.80
VAL_RATIO               = 0.10
TEST_RATIO              = 0.10

# ── ImageNet normalization stats ────────────────────────────────────────────────
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# ── 28 official development classes (canonical names) ──────────────────────────
DEVELOPMENT_CLASSES = [
    'Apple — Apple Scab',
    'Apple — Healthy',
    'Apple — Cedar Apple Rust',
    'Blueberry — Healthy',
    'Cherry — Healthy',
    'Corn — Cercospora Leaf Spot / Gray Leaf Spot',
    'Corn — Common Rust',
    'Corn — Northern Leaf Blight',
    'Grape — Black Rot',
    'Grape — Healthy',
    'Peach — Healthy',
    'Bell Pepper — Bacterial Spot',
    'Bell Pepper — Healthy',
    'Potato — Early Blight',
    'Potato — Late Blight',
    'Raspberry — Healthy',
    'Soybean — Healthy',
    'Squash — Powdery Mildew',
    'Strawberry — Healthy',
    'Tomato — Bacterial Spot',
    'Tomato — Early Blight',
    'Tomato — Late Blight',
    'Tomato — Leaf Mold',
    'Tomato — Septoria Leaf Spot',
    'Tomato — Spider Mites / Two-Spotted Spider Mite',
    'Tomato — Tomato Yellow Leaf Curl Virus',
    'Tomato — Tomato Mosaic Virus',
    'Tomato — Healthy',
]

print(f'Config loaded.')
print(f'  ROOT:    {ROOT}')
print(f'  RAW:     {RAW_DIR}')
print(f'  TRAIN:   {TRAIN_DIR}')
print(f'  VAL:     {VAL_DIR}')
print(f'  TEST:    {TEST_DIR}')
print(f'  Classes: {len(DEVELOPMENT_CLASSES)}')

## Cell 4 — Class Mapping Table + Sanitization Helper

**CRITICAL:** Two development class names contain `/` which is a Linux path separator.  
We sanitize class names for use as filesystem folder names by replacing `/` with `_`.  
The canonical class names (with `/`) are preserved separately and stored in the model checkpoint.

In [ ]:
def sanitize_folder_name(class_name: str) -> str:
    """
    Convert a canonical development class name to a filesystem-safe folder name.
    Rule: replace '/' with '_'  (matches local _match_class logic)
    """
    return class_name.replace('/', '_')


def folder_to_canonical(folder_name: str) -> str:
    """
    Convert a sanitized folder name back to the canonical development class name.
    Inverse of sanitize_folder_name().
    """
    folder_to_dev = {sanitize_folder_name(dc): dc for dc in DEVELOPMENT_CLASSES}
    return folder_to_dev.get(folder_name, folder_name)


# Build lookup: sanitized folder name -> canonical name
FOLDER_TO_CANONICAL = {sanitize_folder_name(dc): dc for dc in DEVELOPMENT_CLASSES}
# Reverse: canonical -> sanitized
CANONICAL_TO_FOLDER = {dc: sanitize_folder_name(dc) for dc in DEVELOPMENT_CLASSES}

# Exact 38->28 mapping: PlantVillage folder name -> canonical development class name (or None=excluded)
CLASS_MAPPING = {
    # Apple
    'Apple___Apple_scab':                                 'Apple — Apple Scab',
    'Apple___Black_rot':                                  None,   # excluded
    'Apple___Cedar_apple_rust':                           'Apple — Cedar Apple Rust',
    'Apple___healthy':                                    'Apple — Healthy',
    # Blueberry
    'Blueberry___healthy':                                'Blueberry — Healthy',
    # Cherry
    'Cherry_(including_sour)___Powdery_mildew':           None,   # excluded
    'Cherry_(including_sour)___healthy':                  'Cherry — Healthy',
    # Corn
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Corn — Cercospora Leaf Spot / Gray Leaf Spot',
    'Corn_(maize)___Common_rust_':                        'Corn — Common Rust',
    'Corn_(maize)___Northern_Leaf_Blight':                'Corn — Northern Leaf Blight',
    'Corn_(maize)___healthy':                             None,   # excluded
    # Grape
    'Grape___Black_rot':                                  'Grape — Black Rot',
    'Grape___Esca_(Black_Measles)':                       None,   # excluded
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)':         None,   # excluded
    'Grape___healthy':                                    'Grape — Healthy',
    # Orange
    'Orange___Haunglongbing_(Citrus_greening)':           None,   # excluded
    # Peach
    'Peach___Bacterial_spot':                             None,   # excluded
    'Peach___healthy':                                    'Peach — Healthy',
    # Pepper
    'Pepper,_bell___Bacterial_spot':                      'Bell Pepper — Bacterial Spot',
    'Pepper,_bell___healthy':                             'Bell Pepper — Healthy',
    # Potato
    'Potato___Early_blight':                              'Potato — Early Blight',
    'Potato___Late_blight':                               'Potato — Late Blight',
    'Potato___healthy':                                   None,   # excluded
    # Raspberry
    'Raspberry___healthy':                                'Raspberry — Healthy',
    # Soybean
    'Soybean___healthy':                                  'Soybean — Healthy',
    # Squash
    'Squash___Powdery_mildew':                            'Squash — Powdery Mildew',
    # Strawberry
    'Strawberry___Leaf_scorch':                           None,   # excluded
    'Strawberry___healthy':                               'Strawberry — Healthy',
    # Tomato
    'Tomato___Bacterial_spot':                            'Tomato — Bacterial Spot',
    'Tomato___Early_blight':                              'Tomato — Early Blight',
    'Tomato___Late_blight':                               'Tomato — Late Blight',
    'Tomato___Leaf_Mold':                                 'Tomato — Leaf Mold',
    'Tomato___Septoria_leaf_spot':                        'Tomato — Septoria Leaf Spot',
    'Tomato___Spider_mites Two-spotted_spider_mite':      'Tomato — Spider Mites / Two-Spotted Spider Mite',
    'Tomato___Target_Spot':                               None,   # excluded
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus':             'Tomato — Tomato Yellow Leaf Curl Virus',
    'Tomato___Tomato_mosaic_virus':                       'Tomato — Tomato Mosaic Virus',
    'Tomato___healthy':                                   'Tomato — Healthy',
}

included = {k: v for k, v in CLASS_MAPPING.items() if v is not None}
excluded = {k for k, v in CLASS_MAPPING.items() if v is None}
print(f'PlantVillage folders in mapping: {len(CLASS_MAPPING)}')
print(f'Mapped to development classes:   {len(included)}')
print(f'Excluded (None):                 {len(excluded)}')
assert len(set(included.values())) == 28, f'Expected 28 unique dev classes, got {len(set(included.values()))}'
print('Mapping table sanity check passed.')

## Cell 5 — Download PlantVillage Dataset from Hugging Face (Direct data.zip)

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

print('Downloading PlantVillage data.zip from Hugging Face (mohanty/PlantVillage)...')
zip_path = hf_hub_download(
    repo_id='mohanty/PlantVillage',
    filename='data.zip',
    repo_type='dataset'
)
print(f'Downloaded archive: {zip_path}')

# Clear and recreate RAW_DIR
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Extracting PlantVillage COLOR/RGB images (raw/color/) ...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    # Select only PlantVillage color/RGB image entries
    color_members = [
        m for m in zf.namelist()
        if m.startswith('raw/color/') and not m.endswith('/')
    ]
    print(f'Found {len(color_members)} RGB images in archive.')
    
    extracted_count = 0
    for member in tqdm(color_members, desc='Extracting raw color images'):
        # Strip 'raw/color/' prefix to maintain PlantVillage class folder hierarchy
        rel_path = os.path.relpath(member, 'raw/color')
        dest_path = RAW_DIR / rel_path
        dest_path.parent.mkdir(parents=True, exist_ok=True)
        
        with zf.open(member) as src, open(dest_path, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        extracted_count += 1

print(f'\nExtraction complete: {extracted_count} raw RGB images extracted to {RAW_DIR}')

## Cell 6 — Inspect Downloaded Raw Dataset

In [ ]:
raw_class_dirs = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
total_raw_imgs = sum(len(list(d.glob('*.*'))) for d in raw_class_dirs)

print(f'Raw PlantVillage Class Folders: {len(raw_class_dirs)}')
print(f'Total Raw Images Discovered:    {total_raw_imgs}')
print('\nFirst 5 class folders:')
for d in raw_class_dirs[:5]:
    count = len(list(d.glob('*.*')))
    print(f'  {d.name:<55} {count:>5} images')

assert len(raw_class_dirs) == 38, f'Expected 38 raw class folders, got {len(raw_class_dirs)}'
assert total_raw_imgs == 54305, f'Expected 54305 raw images, got {total_raw_imgs}'
print('\nRaw dataset inspection PASSED: 38 folders, 54,305 total images.')

## Cell 7 — Apply 38→28 Class Mapping

**Key fix:** class names with `/` (path separator on Linux) are sanitized to `_` for filesystem use.  
Original filenames containing physical `___<leaf_id>` tags are preserved intact for group splitting.

In [ ]:
# Clear and recreate mapped dir
if MAPPED_DIR.exists():
    shutil.rmtree(MAPPED_DIR)
MAPPED_DIR.mkdir(parents=True, exist_ok=True)

mapped_count  = 0
skipped_count = 0
unmapped      = []

for raw_class_dir in sorted(RAW_DIR.iterdir()):
    if not raw_class_dir.is_dir():
        continue
    pv_folder = raw_class_dir.name   # PlantVillage folder name

    canonical = CLASS_MAPPING.get(pv_folder)
    if canonical is None:
        # Excluded class
        imgs = list(raw_class_dir.glob('*.*'))
        skipped_count += len(imgs)
        continue

    folder_name = sanitize_folder_name(canonical)
    dest_dir    = MAPPED_DIR / folder_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    imgs = [f for f in raw_class_dir.glob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    for img_path in imgs:
        dest = dest_dir / img_path.name   # Preserve original filename containing ___<leaf_id>
        shutil.copy2(str(img_path), str(dest))
        mapped_count += 1

print(f'Mapping complete.')
print(f'  Mapped images:   {mapped_count}')
print(f'  Excluded images: {skipped_count}')

mapped_dirs = sorted([d for d in MAPPED_DIR.iterdir() if d.is_dir()])
print(f'\nMapped class folders on disk: {len(mapped_dirs)}')
assert len(mapped_dirs) == 28, f'Expected 28 mapped class folders, got {len(mapped_dirs)}'
assert mapped_count == 38542, f'Expected 38542 mapped images, got {mapped_count}'
print('\nMapping sanity check PASSED: 28 development class folders present.')

## Cell 8 — Create 80/10/10 Leaf-Group Isolated Split

Extracts `leaf_id` from PlantVillage filenames (`___<leaf_id>`) so that multi-view photos of the same physical leaf stay together in the same split (train, val, or test).

In [ ]:
import random
import re
import os
import shutil
import numpy as np
from collections import defaultdict


def extract_leaf_group_id(filename: str) -> str:
    """
    Extract physical leaf / photo session group ID from PlantVillage filename.
    Guarantees leaf-level leakage safety across train/val/test splits.
    """
    base = os.path.splitext(filename)[0]
    if '___' in base:
        right_part = base.split('___')[1]
    else:
        right_part = base
    
    clean_id = re.sub(r'\s+copy(\s+\d+)?$', '', right_part, flags=re.IGNORECASE)
    clean_id = re.sub(r'\s*\(\d+\)$', '', clean_id)
    return clean_id.strip()


random.seed(SEED)
np.random.seed(SEED)

# Clear existing splits
for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if split_dir.exists():
        shutil.rmtree(split_dir)

# Group mapped files by development class and leaf group ID
class_leaf_groups = defaultdict(lambda: defaultdict(list))
total_mapped_imgs = 0

for cls_dir in sorted(MAPPED_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    folder_name = cls_dir.name      # sanitized folder name
    
    for img_path in sorted(cls_dir.glob('*.*')):
        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            leaf_id = extract_leaf_group_id(img_path.name)
            class_leaf_groups[folder_name][leaf_id].append(img_path)
            total_mapped_imgs += 1

print(f'Grouped {total_mapped_imgs} images into physical leaf groups across {len(class_leaf_groups)} classes.')

split_stats = {'train': {}, 'val': {}, 'test': {}}
split_file_paths = {'train': set(), 'val': set(), 'test': set()}
split_leaf_groups = {'train': set(), 'val': set(), 'test': set()}
total = {'train': 0, 'val': 0, 'test': 0}
SPLIT_DIRS = {'train': TRAIN_DIR, 'val': VAL_DIR, 'test': TEST_DIR}

for idx, folder_name in enumerate(sorted(class_leaf_groups.keys())):
    leaf_dict = class_leaf_groups[folder_name]
    group_keys = sorted(list(leaf_dict.keys()))
    
    # Deterministic class-level shuffle using SEED + class_idx
    rng = random.Random(SEED + idx)
    rng.shuffle(group_keys)

    total_class_imgs = sum(len(leaf_dict[g]) for g in group_keys)
    target_train = total_class_imgs * TRAIN_RATIO
    target_val   = total_class_imgs * VAL_RATIO

    curr_train = 0
    curr_val   = 0

    for g_key in group_keys:
        imgs   = leaf_dict[g_key]
        n_imgs = len(imgs)

        if curr_train < target_train or (curr_train == 0):
            target_split = 'train'
            curr_train += n_imgs
        elif curr_val < target_val or (curr_val == 0):
            target_split = 'val'
            curr_val += n_imgs
        else:
            target_split = 'test'

        dest_dir = SPLIT_DIRS[target_split] / folder_name
        dest_dir.mkdir(parents=True, exist_ok=True)

        for img in imgs:
            shutil.copy2(str(img), str(dest_dir / img.name))
            split_file_paths[target_split].add(img.name)
            split_leaf_groups[target_split].add(f'{folder_name}::{g_key}')

        split_stats[target_split][folder_name] = split_stats[target_split].get(folder_name, 0) + n_imgs
        total[target_split] += n_imgs

print('\nLeaf-Group Isolated Split Complete:')
print(f'  Train: {total["train"]:>6} images ({total["train"]/total_mapped_imgs:.2%})')
print(f'  Val:   {total["val"]:>6} images ({total["val"]/total_mapped_imgs:.2%})')
print(f'  Test:  {total["test"]:>6} images ({total["test"]/total_mapped_imgs:.2%})')
print(f'  Total: {sum(total.values()):>6} images')

## Cell 9 — Verify Processed Dataset & Leakage Safety

Counts classes and images in every split. Verifies 0 file overlap and 0 physical leaf group overlap.

In [ ]:
print('=' * 70)
print('PROCESSED DATASET INTEGRITY & LEAKAGE VERIFICATION')
print('=' * 70)

# 1. File overlap check
train_val_file_overlap  = split_file_paths['train'] & split_file_paths['val']
train_test_file_overlap = split_file_paths['train'] & split_file_paths['test']
val_test_file_overlap   = split_file_paths['val'] & split_file_paths['test']
total_file_overlap      = len(train_val_file_overlap) + len(train_test_file_overlap) + len(val_test_file_overlap)

# 2. Leaf group overlap check
train_val_leaf_overlap  = split_leaf_groups['train'] & split_leaf_groups['val']
train_test_leaf_overlap = split_leaf_groups['train'] & split_leaf_groups['test']
val_test_leaf_overlap   = split_leaf_groups['val'] & split_leaf_groups['test']
total_leaf_overlap      = len(train_val_leaf_overlap) + len(train_test_leaf_overlap) + len(val_test_leaf_overlap)

print(f'File Overlap across splits:       {total_file_overlap}')
print(f'Leaf Group Overlap across splits: {total_leaf_overlap}')

assert total_file_overlap == 0, f'CRITICAL DATA LEAKAGE: {total_file_overlap} files overlap across splits!'
assert total_leaf_overlap == 0, f'CRITICAL DATA LEAKAGE: {total_leaf_overlap} physical leaf groups overlap across splits!'

for split_name, split_dir in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    print(f'\n[{split_name.upper()}]  {split_dir}')
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    empty   = []
    missing = []
    total_n = 0

    for dc in DEVELOPMENT_CLASSES:
        fn = sanitize_folder_name(dc)
        d  = split_dir / fn
        if not d.exists():
            missing.append(fn)
            continue
        n = len(list(d.glob('*.*')))
        total_n += n
        if n == 0:
            empty.append(fn)
        print(f'    {fn:<60}  {n:>5} imgs')

    print(f'  Total images: {total_n}')
    assert not missing, f'Missing folders in {split_name}: {missing}'
    assert not empty, f'Empty folders in {split_name}: {empty}'

print('\n' + '=' * 70)
print('VERIFICATION PASSED: All 28 classes present in train/val/test with ZERO data leakage.')
print('=' * 70)

## Cell 10 — Diagnostic: Resolved Paths and Directory State

In [ ]:
print('RESOLVED PATHS')
print('-' * 60)
for name, path in [
    ('ROOT',          ROOT),
    ('RAW_DIR',       RAW_DIR),
    ('MAPPED_DIR',    MAPPED_DIR),
    ('TRAIN_DIR',     TRAIN_DIR),
    ('VAL_DIR',       VAL_DIR),
    ('TEST_DIR',      TEST_DIR),
    ('MODELS_DIR',    MODELS_DIR),
    ('BEST_MODEL',    BEST_MODEL_PATH),
    ('CLASSES_JSON',  CLASSES_JSON),
]:
    exists = '  EXISTS' if path.exists() else '  MISSING'
    print(f'  {name:<15} {str(path):<55} {exists}')

print()
print('TRAIN_DIR class subfolder count:', len([d for d in TRAIN_DIR.iterdir() if d.is_dir()]) if TRAIN_DIR.exists() else 'DIR MISSING')
print('VAL_DIR   class subfolder count:', len([d for d in VAL_DIR.iterdir()   if d.is_dir()]) if VAL_DIR.exists()   else 'DIR MISSING')
print('TEST_DIR  class subfolder count:', len([d for d in TEST_DIR.iterdir()  if d.is_dir()]) if TEST_DIR.exists()  else 'DIR MISSING')

## Cell 11 — Assert Processed Dataset Before ImageFolder

Hard stop if TRAIN_DIR is missing, empty, or has fewer than 28 class subfolders.

In [ ]:
def assert_split_ready(split_name: str, split_dir: Path, expected_classes: list):
    """Raise AssertionError with a clear message if the split directory is not ready."""
    assert split_dir.exists(), (
        f'SPLIT NOT READY: {split_name} directory does not exist: {split_dir}\n'
        f'Run Cells 7 and 8 (apply mapping + create split) before proceeding.'
    )
    class_subdirs = [d for d in split_dir.iterdir() if d.is_dir()]
    assert len(class_subdirs) > 0, (
        f'SPLIT NOT READY: {split_name} directory exists but contains no class subfolders.\n'
        f'Path: {split_dir}'
    )
    n_expected = len(expected_classes)
    n_got      = len(class_subdirs)
    assert n_got == n_expected, (
        f'SPLIT NOT READY: {split_name} has {n_got} class folders, expected {n_expected}.'
    )
    empty = [d.name for d in class_subdirs if len(list(d.glob('*.*'))) == 0]
    assert not empty, (
        f'SPLIT NOT READY: {split_name} has empty class folders: {empty}'
    )
    n_images = sum(len(list(d.glob('*.*'))) for d in class_subdirs)
    print(f'  [{split_name}] OK: {n_got} classes, {n_images} images')


print('Asserting processed split readiness...')
assert_split_ready('train', TRAIN_DIR, DEVELOPMENT_CLASSES)
assert_split_ready('val',   VAL_DIR,   DEVELOPMENT_CLASSES)
assert_split_ready('test',  TEST_DIR,  DEVELOPMENT_CLASSES)
print('\nAll assertions PASSED. Safe to build DataLoaders.')

## Cell 12 — Define Transforms

In [ ]:
import io
import random
from PIL import Image
import torchvision.transforms as T


class AddGaussianNoise:
    """Adds mild Gaussian noise to image tensor for field sensor robustness."""
    def __init__(self, std=0.03, p=0.2):
        self.std = std
        self.p   = p
    def __call__(self, tensor):
        if random.random() < self.p:
            return torch.clamp(tensor + torch.randn_like(tensor) * self.std, 0., 1.)
        return tensor


class JPEGCompressionDegradation:
    """Simulates JPEG compression artifacts common in low-bandwidth field uploads."""
    def __init__(self, quality_min=40, quality_max=85, p=0.25):
        self.quality_min = quality_min
        self.quality_max = quality_max
        self.p = p
    def __call__(self, img):
        if random.random() < self.p:
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=random.randint(self.quality_min, self.quality_max))
            buf.seek(0)
            return Image.open(buf).convert('RGB')
        return img


# Medium augmentation — matches training/preprocessing.py AUGMENTATION_LEVEL='medium'
train_transform = T.Compose([
    JPEGCompressionDegradation(quality_min=40, quality_max=85, p=0.25),
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
    T.ToTensor(),
    AddGaussianNoise(std=0.03, p=0.2),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

print('Transforms defined (medium augmentation).')
print(f'  Image size: {IMAGE_SIZE} x {IMAGE_SIZE}')

## Cell 13 — Build DataLoaders + Verify Class Mapping

After ImageFolder loads the sanitized folder names, we remap them to canonical development class names.  
This is required so the model checkpoint stores the original names expected by `inference/predict.py`.

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler

# ── Build ImageFolder datasets ─────────────────────────────────────────────────
train_ds = ImageFolder(str(TRAIN_DIR), transform=train_transform)
val_ds   = ImageFolder(str(VAL_DIR),   transform=val_transform)
test_ds  = ImageFolder(str(TEST_DIR),  transform=val_transform)

print(f'ImageFolder loaded.')
print(f'  Train: {len(train_ds)} images,  {len(train_ds.classes)} classes')
print(f'  Val:   {len(val_ds)} images,   {len(val_ds.classes)} classes')
print(f'  Test:  {len(test_ds)} images,  {len(test_ds.classes)} classes')

# ── Remap sanitized folder names -> canonical development class names ───────────
folder_classes = train_ds.classes
class_names    = [FOLDER_TO_CANONICAL.get(fn, fn) for fn in folder_classes]

print(f'\nClass remapping (sanitized folder -> canonical name):')
for fn, cn in zip(folder_classes, class_names):
    marker = '  <-- REMAPPED' if fn != cn else ''
    print(f'  {fn!r:65}  ->  {cn!r}{marker}')

# ── Verify: class_names matches DEVELOPMENT_CLASSES ──
assert set(class_names) == set(DEVELOPMENT_CLASSES), (
    f'CLASS MISMATCH after remapping!\n'
    f'Got:      {sorted(class_names)}\n'
    f'Expected: {sorted(DEVELOPMENT_CLASSES)}'
)
print(f'\nClass mapping verification PASSED.')
print(f'  ImageFolder has exactly the 28 expected development classes.')

# ── Verify val and test use same class order as train ─────────────────────────
assert train_ds.class_to_idx == val_ds.class_to_idx == test_ds.class_to_idx, (
    'Train/val/test class_to_idx differ — class ordering inconsistency!'
)
print(f'  train/val/test class_to_idx are identical. OK')

# ── Weighted sampler for class imbalance ───────────────────────────────────────
class_counts  = torch.zeros(len(class_names))
for _, lbl in train_ds.samples:
    class_counts[lbl] += 1
class_weights  = 1.0 / (class_counts + 1e-6)
class_weights  = class_weights / class_weights.sum() * len(class_names)
sample_weights = [class_weights[lbl].item() for _, lbl in train_ds.samples]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# ── DataLoaders ────────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'\nDataLoaders ready.')
print(f'  Train: {len(train_loader)} batches (batch_size={BATCH_SIZE})')
print(f'  Val:   {len(val_loader)} batches')
print(f'  Test:  {len(test_loader)} batches')

## Cell 14 — Verify DataLoader Batch Shapes and Class Count

In [ ]:
# Pull one batch from each loader and verify shapes
train_imgs, train_lbls = next(iter(train_loader))
val_imgs,   val_lbls   = next(iter(val_loader))

print('Train batch:')
print(f'  images: {list(train_imgs.shape)}  (expected [32, 3, {IMAGE_SIZE}, {IMAGE_SIZE}])')
print(f'  labels: {list(train_lbls.shape)}  unique labels in batch: {sorted(train_lbls.unique().tolist())}')

print('Val batch:')
print(f'  images: {list(val_imgs.shape)}')
print(f'  labels: {list(val_lbls.shape)}')

print(f'\nnum_classes: {len(class_names)}')
print(f'class_names[0]:  {class_names[0]!r}')
print(f'class_names[-1]: {class_names[-1]!r}')

# Hard assertion
assert train_imgs.shape[1:] == torch.Size([3, IMAGE_SIZE, IMAGE_SIZE]), 'Wrong train image shape!'
assert len(class_names) == 28, f'Expected 28 classes, got {len(class_names)}'
print('\nDataLoader shape verification PASSED.')

## Cell 15 — Build Model (EfficientNet-B2)

In [ ]:
import timm
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')

num_classes = len(class_names)
model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=num_classes)
model = model.to(device)

# Loss with inverse-frequency class weights
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Cosine LR scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# AMP GradScaler
scaler = GradScaler('cuda', enabled=USE_AMP)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model:            {MODEL_NAME}')
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Output classes:   {num_classes}')

## Cell 16 — Training Loop (2-Stage: Warmup → Full Fine-Tune)

In [ ]:
import csv
import time
import json
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm as tqdm_auto


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm_auto(loader, desc=f'Epoch {epoch+1} [Train]', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(all_labels, all_preds)
    return avg_loss, macro_f1, accuracy


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm_auto(loader, desc='[Val]', leave=False):
        images, labels = images.to(device), labels.to(device)
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(all_labels, all_preds)
    return avg_loss, macro_f1, accuracy


def log_experiment(row):
    file_exists = EXPERIMENT_LOG.exists()
    with open(EXPERIMENT_LOG, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


# ── Main training loop ──────────────────────────────────────────────────────────
best_val_f1      = 0.0
epochs_no_improve = 0
experiment_id    = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'Starting training  |  model={MODEL_NAME}  |  epochs={NUM_EPOCHS}  |  device={device}')
print('=' * 80)

for epoch in range(NUM_EPOCHS):

    # Stage 1: freeze backbone for WARMUP_EPOCHS
    if epoch == 0:
        print(f'Stage 1: Freezing backbone for {WARMUP_EPOCHS} warmup epochs...')
        for name, param in model.named_parameters():
            if not any(x in name for x in ['classifier', 'head', 'fc']):
                param.requires_grad = False
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params (head only): {n_trainable:,}')

    # Stage 2: unfreeze all
    elif epoch == WARMUP_EPOCHS:
        print(f'Stage 2: Unfreezing all layers for full fine-tuning...')
        for param in model.parameters():
            param.requires_grad = True
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params (all): {n_trainable:,}')

    t0 = time.time()
    train_loss, train_f1, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch)
    val_loss, val_f1, val_acc = validate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f'Epoch {epoch+1:03d}/{NUM_EPOCHS}  '
        f'Train Loss:{train_loss:.4f} F1:{train_f1:.4f} Acc:{train_acc:.4f}  '
        f'Val Loss:{val_loss:.4f} F1:{val_f1:.4f} Acc:{val_acc:.4f}  '
        f'LR:{current_lr:.2e}  {elapsed:.0f}s'
    )

    # Save best model — stores CANONICAL class_names (with '/')
    if val_f1 > best_val_f1:
        best_val_f1       = val_f1
        epochs_no_improve = 0
        torch.save({
            'epoch':           epoch + 1,
            'model_name':      MODEL_NAME,
            'num_classes':     num_classes,
            'class_names':     class_names,   # CANONICAL names (with '/') for inference
            'image_size':      IMAGE_SIZE,
            'state_dict':      model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_macro_f1':    val_f1,
            'val_accuracy':    val_acc,
        }, BEST_MODEL_PATH)
        print(f'  --> Best model saved  (Val Macro-F1: {best_val_f1:.4f})')
    else:
        epochs_no_improve += 1

    # Always save last checkpoint
    torch.save({'epoch': epoch + 1, 'state_dict': model.state_dict()}, LAST_MODEL_PATH)

    # Experiment log
    log_experiment({
        'experiment_id': experiment_id, 'epoch': epoch + 1, 'model': MODEL_NAME,
        'batch_size':    BATCH_SIZE,    'lr':    current_lr,
        'train_loss':    round(train_loss, 4), 'train_f1':  round(train_f1, 4),
        'train_acc':     round(train_acc, 4),  'val_loss':  round(val_loss, 4),
        'val_f1':        round(val_f1, 4),     'val_acc':   round(val_acc, 4),
        'best_val_f1':   round(best_val_f1, 4),
    })

    # Early stopping
    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f'\nEarly stopping triggered after {EARLY_STOPPING_PATIENCE} epochs without improvement.')
        break

print(f'\nTraining complete.  Best Val Macro-F1: {best_val_f1:.4f}')
print(f'Model saved to: {BEST_MODEL_PATH}')

## Cell 17 — Save classes.json

In [ ]:
import json

# Save canonical class index -> name mapping (uses canonical names with '/')
mapping = {str(i): name for i, name in enumerate(class_names)}
with open(CLASSES_JSON, 'w') as f:
    json.dump(mapping, f, indent=2)

print(f'Saved: {CLASSES_JSON}')
print(f'Class mapping ({len(mapping)} entries):')
for k, v in mapping.items():
    print(f'  {k:>2}: {v}')

# Cross-check: classes.json must match model checkpoint
ckpt = torch.load(BEST_MODEL_PATH, map_location='cpu')
assert ckpt['class_names'] == class_names, 'classes.json and checkpoint class list differ!'
print('\nclasses.json matches model checkpoint. OK')

## Cell 18 — Full Evaluation on Validation Set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.metrics import recall_score, precision_score
from collections import Counter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
ckpt       = torch.load(BEST_MODEL_PATH, map_location=device)
eval_model = timm.create_model(ckpt['model_name'], pretrained=False, num_classes=ckpt['num_classes'])
eval_model.load_state_dict(ckpt['state_dict'])
eval_model = eval_model.to(device).eval()
ckpt_classes = ckpt['class_names']    # canonical names

print(f'Loaded checkpoint: epoch={ckpt["epoch"]}  Val Macro-F1={ckpt["val_macro_f1"]:.4f}')

# Collect predictions
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm_auto(val_loader, desc='Evaluating'):
        images = images.to(device)
        with autocast('cuda', enabled=USE_AMP):
            outputs = eval_model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
accuracy  = accuracy_score(all_labels, all_preds)

print(f'\n{"="*60}')
print(f'  Val Macro-F1 : {macro_f1:.4f}')
print(f'  Val Accuracy : {accuracy:.4f}')
print(f'{"="*60}')
print()
print(classification_report(all_labels, all_preds, target_names=ckpt_classes, zero_division=0))

# Confusion matrix
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-6)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=ckpt_classes, yticklabels=ckpt_classes,
    ax=ax, linewidths=0.5, annot_kws={'size': 6},
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual',    fontsize=11)
ax.set_title('Normalised Confusion Matrix — AgriSmart AI (Validation Set)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
cm_path = EVAL_DIR / 'confusion_matrix_val.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix saved: {cm_path}')

# Error analysis
recalls    = recall_score(all_labels, all_preds, average=None, zero_division=0)
precisions = precision_score(all_labels, all_preds, average=None, zero_division=0)
f1s        = f1_score(all_labels, all_preds, average=None, zero_division=0)
class_perf = sorted(zip(ckpt_classes, recalls, precisions, f1s), key=lambda x: x[3])

print('\nBottom 5 classes by F1:')
for name, rec, prec, f1 in class_perf[:5]:
    print(f'  {name:<55}  Recall:{rec:.3f}  Prec:{prec:.3f}  F1:{f1:.3f}')

print('\nTop 5 classes by F1:')
for name, rec, prec, f1 in class_perf[-5:]:
    print(f'  {name:<55}  Recall:{rec:.3f}  Prec:{prec:.3f}  F1:{f1:.3f}')

wrong_mask = all_labels != all_preds
pairs      = Counter(zip(all_labels[wrong_mask].tolist(), all_preds[wrong_mask].tolist()))
print('\nTop 10 confusion pairs (true -> predicted):')
for (t, p), count in pairs.most_common(10):
    print(f'  {ckpt_classes[t]:<50} -> {ckpt_classes[p]:<50}  ({count})')

max_probs    = all_probs.max(axis=1)
correct_mask = all_labels == all_preds
print(f'\nConfidence: correct mean={max_probs[correct_mask].mean():.3f}  wrong mean={max_probs[~correct_mask].mean():.3f}')

# Save evaluation JSON
report_dict = classification_report(
    all_labels, all_preds, target_names=ckpt_classes, output_dict=True, zero_division=0)
result = {
    'split':    'val',
    'macro_f1': round(macro_f1, 4),
    'accuracy': round(accuracy, 4),
    'model':    MODEL_NAME,
    'epoch':    int(ckpt['epoch']),
    'per_class': {
        name: {
            'precision': round(report_dict[name]['precision'], 4),
            'recall':    round(report_dict[name]['recall'], 4),
            'f1':        round(report_dict[name]['f1-score'], 4),
            'support':   int(report_dict[name]['support']),
        }
        for name in ckpt_classes if name in report_dict
    },
}
eval_json = EVAL_DIR / 'evaluation_val.json'
with open(eval_json, 'w') as f:
    json.dump(result, f, indent=2)
print(f'\nEvaluation results saved: {eval_json}')
print(f'Final Val Macro-F1: {macro_f1:.4f}')

## Cell 19 — Download All Results

Downloads model weights, class mapping, experiment log, evaluation JSON, and confusion matrix to your local machine.  
Place them in your local project:

```
agrismart_best.pth       →  D:/Agrismart_AI/models/
classes.json             →  D:/Agrismart_AI/models/
experiments.csv          →  D:/Agrismart_AI/
evaluation_val.json      →  D:/Agrismart_AI/evaluation/results/
confusion_matrix_val.png →  D:/Agrismart_AI/evaluation/results/
```

In [ ]:
from google.colab import files

to_download = [
    (BEST_MODEL_PATH,                  'agrismart_best.pth'),
    (CLASSES_JSON,                     'classes.json'),
    (EXPERIMENT_LOG,                   'experiments.csv'),
    (EVAL_DIR / 'evaluation_val.json', 'evaluation_val.json'),
    (EVAL_DIR / 'confusion_matrix_val.png', 'confusion_matrix_val.png'),
]

for path, label in to_download:
    if path.exists():
        files.download(str(path))
        print(f'Downloaded: {label}')
    else:
        print(f'MISSING (skipped): {label}  ({path})')

print('\nDone. Place files in D:/Agrismart_AI/ as described above.')